#### ENRICH `ORDER_PAYMENTS` FROM BRONZE TO SILVER

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDER_PAYMENTS_ENRICHED'

In [0]:
order_payments_df = spark.table('ORDERS_SALES.BRONZE.ORDERS_PAYMENTS')
order_payments_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

order_payments_trans_df = order_payments_df.withColumn(
    "payments_norm",
    when(
        (col("payment_installments") > 0) & col("payment_value").isNotNull(),
        round(col("payment_value") / col("payment_installments"), 2)
    ).otherwise(None)
)
order_payments_trans_df.limit(1).display()

In [0]:
order_payments_trans_ordered_df = order_payments_trans_df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    col("payment_installments").cast("int").alias("payment_installments"),
    "payment_value",
    "payments_norm",
    "file_name",
    "file_path",
    "load_timestamp"
)
order_payments_trans_ordered_df.limit(1).display()

In [0]:
order_payments_trans_ordered_df.createOrReplaceTempView('order_payments_trans_ordered_df_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED`

In [0]:
spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM order_payments_trans_ordered_df_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id = tgt.order_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_id = src.order_id,
    tgt.payment_type = src.payment_type
WHEN NOT MATCHED THEN
INSERT (order_id, payment_sequential, payment_type, payment_installments, payment_value, payments_norm, origin_file_path,
load_timestamp)
VALUES (src.order_id, src.payment_sequential, src.payment_type, src.payment_installments, src.payment_value, src.payments_norm, src.file_path, current_timestamp)
''')

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED')